# EU Laboratory Report — Part 6: From FHIR Messages to a FHIR Document

This is the sixth notebook in the series. Every notebook so far has built a FHIR
**Message** `Bundle` (`Bundle.type = "message"`) — a `MessageHeader` plus the resources
one system hands another. This notebook builds a FHIR **Document** instead
(`Bundle.type = "document"`, led by a `Composition`) — the newest version of the idea
Clinical Document Architecture (CDA) captured, and the format NHS England's Unified
Genomic Record (UGR) Phase 2 and the [NHS Pathology FHIR Implementation
Guide](https://simplifier.net/guide/pathology-fhir-implementation-guide) expect. Readers
who know [NHS England's Transfer of Care STU3 FHIR
API](https://digital.nhs.uk/developer/api-catalogue/transfer-of-care) will recognise the
shape: a FHIR Document is that same idea's FHIR-native, R4 successor.

A `Composition` gives a document two things a plain Message `Bundle` doesn't have:

- **A fixed clinical structure** — `section`s, each coded, each optionally nested,
  each pointing (`entry`) at the structured resources that back it.
- **Human-readable narrative** (HTML) at both the whole-document and per-section level —
  so a system that only knows how to render HTML, and one that parses `Observation`s
  programmatically, can both consume the *same* document.

Two worked examples, reusing everything already built earlier in this series rather than
converting from HL7 v2 a third time:

- **Genomics** — `04-laboratory-report-fhir-from-hl7v2.ipynb`'s **Laboratory Report**
  (narrative PDF, `DiagnosticReport`/`DocumentReference`/`Binary`) merged with
  `05-test-results-from-vcf.ipynb`'s **Test Results** (four `variant` `Observation`s) into
  one document: the variants become `DiagnosticReport.result` entries on the Laboratory
  Report's own `DiagnosticReport`; the Test Results message's separate `DiagnosticReport`
  — which only ever existed to carry those Observations before a report existed to
  attach them to — is discarded once its job is done.
- **Pathology** — much simpler: there's no separate discrete-results message to merge in,
  so it's just the Laboratory Report's `DiagnosticReport` wrapped in a `Composition`.

The payoff of building a `Composition` at all, rather than stopping at a plain Message
`Bundle`, is that a document with per-section HTML narrative can be *rendered* — section
9 below takes the genomics `Bundle` this notebook builds and turns it into an actual
styled HTML page, viewable in a browser, generated entirely from the `Composition`'s own
`section[].text` content plus a small CSS stylesheet.

Validation now checks against a third IG alongside the two already in use
(`package.tgz` for NW-GMSA, `hl7.fhir.uv.genomics-reporting#3.0.0` for the variants): the
**EU Laboratory IG** (`hl7.fhir.eu.laboratory#2.0.0`, canonical
`http://hl7.eu/fhir/laboratory`), already present in this machine's local FHIR package
cache. Its own published example, `Composition-comp-lab-example.json`, is what the
`Composition` built below is modelled on directly, field for field, rather than guessed
from the profile alone.

GitHub's notebook viewer doesn't render ```mermaid``` code fences (unlike GitHub's
regular Markdown-file viewer) — this diagram is embedded instead as an image from
[mermaid.ink](https://mermaid.ink) (base64-encoded diagram source in the URL itself, no
server-side state), which renders the same on GitHub, in Jupyter, and anywhere else that
can display an `<img>`. Source kept alongside for anyone editing the diagram.

![notebook 6 flow](https://mermaid.ink/svg/Zmxvd2NoYXJ0IFRCCiAgICBWMlsiSW5wdXQvVjIvUjAxL2N0ZG5hOTczNzM4MzIyMi50eHQKKHNhbWUgc291cmNlIG1lc3NhZ2UgYXMgMDQgYW5kIDA1KSJdCiAgICBWQ0ZbIklucHV0L0RTUy9WQ0YvaWdlbmVfZXhhbXBsZV9kYXRhLnZjZgooc2FtZSBzb3VyY2UgVkNGIGFzIDA1KSJdCgogICAgVjIgLS0+fCJyZS1kZXJpdmUgcmVwb3J0X3JvdwooMiwgc2FtZSBhcyAwNC8wNSkifCBMQUJSRVBPUlRbIkxhYm9yYXRvcnkgUmVwb3J0IHJlc291cmNlcwpQYXRpZW50IC8gRW5jb3VudGVyIC8gU2VydmljZVJlcXVlc3QKT2JzZXJ2YXRpb24gKHBhbmVsKSAvIEJpbmFyeSAvIERvY3VtZW50UmVmZXJlbmNlCkRpYWdub3N0aWNSZXBvcnQgICgzLCBzYW1lIGFzIDA0KSJdCiAgICBWQ0YgLS0+fCJidWlsZF9vYnNlcnZhdGlvbigpCig0LCB1bmNoYW5nZWQgZnJvbSAwNSkifCBWQVJJQU5UU1siNHggdmFyaWFudCBPYnNlcnZhdGlvbgooYnVpbHQgZGlyZWN0bHkgYWdhaW5zdCAwNCdzIFBhdGllbnQgLQpubyBzZXBhcmF0ZSB1dWlkcyB0byByZWNvbmNpbGUpIl0KCiAgICBMQUJSRVBPUlQgLS0+IE1FUkdFWyJNZXJnZSAoNikKYXBwZW5kIHZhcmlhbnRzIHRvIERpYWdub3N0aWNSZXBvcnQucmVzdWx0CmFkZCBnZW5vbWljLXJlcG9ydCBwcm9maWxlICsgTE9JTkMgNTE5NjktNCJdCiAgICBWQVJJQU5UUyAtLT4gTUVSR0UKCiAgICBNRVJHRSAtLT4gQ09NUEFbIkNvbXBvc2l0aW9uIC0gR2Vub21pY3MgKDcpCjIgc2VjdGlvbnM6IExhYm9yYXRvcnkgUmVwb3J0LCBHZW5vbWljIEZpbmRpbmdzIl0KICAgIENPTVBBIC0tPiBET0NBWyJEb2N1bWVudCBCdW5kbGUgQQpjdGRuYTk3MzczODMyMjItZXVsYWItZG9jdW1lbnQuanNvbiAoOCkiXQogICAgRE9DQSAtLT4gUkVOREVSWyJSZW5kZXIgYXMgSFRNTCArIENTUyAoOSkKZG9jdW1lbnRfdG9faHRtbCgpIl0KCiAgICBMQUJSRVBPUlQgLS0+fCJubyBtZXJnZSBuZWVkZWQifCBDT01QQlsiQ29tcG9zaXRpb24gLSBQYXRob2xvZ3kgKDExKQoxIHNlY3Rpb246IExhYm9yYXRvcnkgUmVwb3J0Il0KICAgIENPTVBCIC0tPiBET0NCWyJEb2N1bWVudCBCdW5kbGUgQgpjdGRuYTk3MzczODMyMjItcGF0aG9sb2d5LWRvY3VtZW50Lmpzb24gKDEyKSJdCg==)

<details><summary>Diagram source</summary>

```mermaid
flowchart TB
    V2["Input/V2/R01/ctdna9737383222.txt
(same source message as 04 and 05)"]
    VCF["Input/DSS/VCF/igene_example_data.vcf
(same source VCF as 05)"]

    V2 -->|"re-derive report_row
(2, same as 04/05)"| LABREPORT["Laboratory Report resources
Patient / Encounter / ServiceRequest
Observation (panel) / Binary / DocumentReference
DiagnosticReport  (3, same as 04)"]
    VCF -->|"build_observation()
(4, unchanged from 05)"| VARIANTS["4x variant Observation
(built directly against 04's Patient -
no separate uuids to reconcile)"]

    LABREPORT --> MERGE["Merge (6)
append variants to DiagnosticReport.result
add genomic-report profile + LOINC 51969-4"]
    VARIANTS --> MERGE

    MERGE --> COMPA["Composition - Genomics (7)
2 sections: Laboratory Report, Genomic Findings"]
    COMPA --> DOCA["Document Bundle A
ctdna9737383222-eulab-document.json (8)"]
    DOCA --> RENDER["Render as HTML + CSS (9)
document_to_html()"]

    LABREPORT -->|"no merge needed"| COMPB["Composition - Pathology (11)
1 section: Laboratory Report"]
    COMPB --> DOCB["Document Bundle B
ctdna9737383222-pathology-document.json (12)"]
```

</details>

In [1]:
import copy
import json
import os
import subprocess
import tempfile
from datetime import datetime
from pathlib import Path
from uuid import uuid4

import pandas as pd
from dotenv import load_dotenv

load_dotenv()

NHS_NUMBER_SYSTEM = "https://fhir.nhs.uk/Id/nhs-number"
ODS_SYSTEM = "https://fhir.nhs.uk/Id/ods-organization-code"
V2_0203 = "http://terminology.hl7.org/CodeSystem/v2-0203"
IGENE_REPORT_ID_SYSTEM = "https://fhir.nwgenomics.nhs.uk/iGene/ReportIdentifier"
GENOMIC_TEST_DIRECTORY_SYSTEM = "https://fhir.nhs.uk/CodeSystem/England-GenomicTestDirectory"
IGEAP_SYSTEM = "https://fhir.nwgenomics.nhs.uk/CodeSystem/IGEAP"
GENOMIC_CLINICAL_INDICATION_SYSTEM = "https://fhir.nwgenomics.nhs.uk/CodeSystem/GenomicClinicalIndication"
GLH_ODS = "699X0"
GLH_NAME = "NHS North West Genomics"

DIAGNOSTIC_REPORT_PROFILE = "https://fhir.nwgenomics.nhs.uk/StructureDefinition/DiagnosticReport"
NWGMSA_OBSERVATION_PROFILE = "https://fhir.nwgenomics.nhs.uk/StructureDefinition/Observation"
SPECIMEN_PROFILE = "https://fhir.nwgenomics.nhs.uk/StructureDefinition/Specimen"
GENOMICS_VARIANT_PROFILE = "http://hl7.org/fhir/uv/genomics-reporting/StructureDefinition/variant"
GENOMICS_REPORT_PROFILE = "http://hl7.org/fhir/uv/genomics-reporting/StructureDefinition/genomic-report"
GENOMICS_REPORTING_IG = "hl7.fhir.uv.genomics-reporting#3.0.0"

EU_LAB_IG = "hl7.fhir.eu.laboratory#2.0.0"
COMPOSITION_EU_LAB_PROFILE = "http://hl7.eu/fhir/laboratory/StructureDefinition/Composition-eu-lab"
BUNDLE_EU_LAB_PROFILE = "http://hl7.eu/fhir/laboratory/StructureDefinition/Bundle-eu-lab"
DIAGNOSTIC_REPORT_REFERENCE_EXT = "http://hl7.eu/fhir/extensions/StructureDefinition/composition-diagnosticReportReference"

os.makedirs("Input/FHIR/R01", exist_ok=True)
os.makedirs("Results/FHIR/R01", exist_ok=True)


def details(item):
    return item["text"]


def issues_df(outcome):
    df = pd.DataFrame(outcome["issue"])
    df["details"] = df["details"].apply(details)
    df.drop(columns=["extension"], inplace=True, errors="ignore")
    df.sort_values(by=["severity"], inplace=True)
    df = df[~df["details"].str.contains("ValueSet/mimetypes")]
    df = df[~df["details"].str.contains("failed: dom-6")]
    df = df[~df["details"].str.contains("bcp:13")]
    df = df[~df["details"].str.contains("no terminology service")]
    return df


def validate(path, igs, profiles, output):
    # Same helper as 05-test-results-from-vcf.ipynb - a list of IGs (local .tgz paths or
    # cached package ids) and profile canonical URLs, rather than 04's single-profile version.
    args = ["java", "-jar", "validator_cli.jar", str(path), "-version", "4.0.1", "-tx", "n/a"]
    for ig in igs:
        args += ["-ig", ig]
    for profile in profiles:
        args += ["-profile", profile]
    args += ["-output", str(output), "-output-style", "json"]
    subprocess.run(args, capture_output=True)
    with open(output) as f:
        return issues_df(json.load(f))


def validate_resource(resource, igs, profiles):
    with tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False) as tmp:
        json.dump(resource, tmp)
        tmp_path = Path(tmp.name)
    return validate(tmp_path, igs, profiles, Path(str(tmp_path) + "-OperationOutcome.json"))


def validate_bundle(path, igs, bundle_checks, output):
    # bundle_checks: list of ("ResourceType:index", profile) pairs, one -bundle flag each -
    # the same repeatable-flag pattern 05 used for its own whole-bundle check.
    args = ["java", "-jar", "validator_cli.jar", str(path), "-version", "4.0.1", "-tx", "n/a"]
    for ig in igs:
        args += ["-ig", ig]
    for locator, profile in bundle_checks:
        args += ["-bundle", locator, profile]
    args += ["-output", str(output), "-output-style", "json"]
    subprocess.run(args, capture_output=True)
    with open(output) as f:
        return issues_df(json.load(f))


def entry_index(bundle, resource):
    return next(i for i, e in enumerate(bundle["entry"]) if e["resource"] is resource)


def component_value(observation, loinc_code):
    # Pull one component's human-readable value off a variant Observation by its LOINC
    # code - used only to render the HTML narrative table in section 6, not to change
    # any resource content.
    for comp in observation.get("component", []):
        if comp["code"]["coding"][0]["code"] != loinc_code:
            continue
        if "valueCodeableConcept" in comp:
            cc = comp["valueCodeableConcept"]
            coding = cc.get("coding", [{}])[0]
            return coding.get("display") or cc.get("text") or coding.get("code", "")
        if "valueString" in comp:
            return comp["valueString"]
        if "valueQuantity" in comp:
            q = comp["valueQuantity"]
            return f"{q['value']} {q.get('unit', q.get('code', ''))}".strip()
    return ""

## Part A — Genomics: merging the Laboratory Report and the Test Results

### 1. Re-derive `report_row` from the source v2 message

Every notebook in this series re-derives what it needs from the *original* source files
rather than reading a previous notebook's saved JSON output — the same choice 05 made
for this exact field (see its section 2), so that any notebook here can be run on its
own, in any order. This cell is unchanged from 04/05 (see 04's section 3 for the
per-field rationale).

In [2]:
def component_field(field, index, default=""):
    parts = field.split("^")
    return parts[index].strip() if index < len(parts) else default


with open("Input/V2/R01/ctdna9737383222.txt", newline="") as f:
    raw_v2 = f.read()

segments = [s for s in raw_v2.replace("\r\n", "\r").split("\r") if s]
by_segment = {}
for segment in segments:
    fields = segment.split("|")
    by_segment.setdefault(fields[0], []).append(fields)

pid = by_segment["PID"][0]
pv1 = by_segment["PV1"][0]
orc = by_segment["ORC"][0]
obr = by_segment["OBR"][0]
obx_list = by_segment["OBX"]
nte = by_segment["NTE"][0]

test_directory_code = nte[3].split("=")[0]
pdf_obx = next(o for o in obx_list if o[4] == "PDF")
outcome_obx = next(o for o in obx_list if o[3].startswith("51968-6"))

report_row = {
    "nhs_number": pid[2],
    "mrn": component_field(pid[3], 0),
    "mrn_assigner_ods": component_field(pid[3], 3),
    "family_name": component_field(pid[5], 0),
    "given_name": component_field(pid[5], 1),
    "birth_date": datetime.strptime(pid[7], "%Y%m%d").strftime("%Y-%m-%d"),
    "sex": pid[8],
    "postcode": component_field(pid[11], 4),
    "account_number": pv1[19],
    "placer_order_number": orc[2],
    "ordering_org_name": component_field(orc[21], 0),
    "ordering_org_ods": component_field(orc[21], 2),
    "filler_report_number": obr[3],
    "test_code_local": component_field(obr[4], 0),
    "test_description": component_field(obr[4], 1),
    "report_datetime": datetime.strptime(obr[22], "%Y%m%d%H%M%S").strftime("%Y-%m-%dT%H:%M:%S+00:00"),
    "specimen_received_datetime": datetime.strptime(obr[14], "%Y%m%d%H%M").strftime("%Y-%m-%dT%H:%M:%S+00:00"),
    "result_status": obr[25],
    "interpreter_family": component_field(obr[32], 1),
    "interpreter_given": component_field(obr[32], 2),
    "test_directory_code": test_directory_code,
    "clinical_indication_code": test_directory_code.split(".")[0],
    "pdf_content_type": component_field(pdf_obx[5], 2),
    "pdf_base64": component_field(pdf_obx[5], 4),
    "outcome_code": component_field(outcome_obx[5], 0),
    "outcome_display": component_field(outcome_obx[5], 1),
}
{k: (v[:40] + "..." if k == "pdf_base64" else v) for k, v in report_row.items()}

{'nhs_number': '9737383222',
 'mrn': 'RXR0817610',
 'mrn_assigner_ods': 'RR8',
 'family_name': 'LEEDS',
 'given_name': 'Rob',
 'birth_date': '1978-01-17',
 'sex': 'M',
 'postcode': 'LS1 3EX',
 'account_number': 'SP26-01847',
 'placer_order_number': '1234-RR8',
 'ordering_org_name': 'Leeds Teaching Hospitals NHS Trust',
 'ordering_org_ods': 'RR8',
 'filler_report_number': 'T26-59X2',
 'test_code_local': 'ctDNA_M4',
 'test_description': 'PACKAGE: M4.14 - Non-Small Cell Lung Cancer, Multi-target ctDNA combined Multi-target NGS panel - small variant (EGFR, ALK, BRAF, KRAS, MET exon 14 skipping and copy number variations) and structural variant (ROS1, RET, ALK, NTRK1, NTRK2, NTRK3, MET exon',
 'report_datetime': '2026-07-14T15:59:16+00:00',
 'specimen_received_datetime': '2026-07-06T00:00:00+00:00',
 'result_status': 'F',
 'interpreter_family': 'Edgerley',
 'interpreter_given': 'Jonathan',
 'test_directory_code': 'M4.14',
 'clinical_indication_code': 'M4',
 'pdf_content_type': 'application/

### 2. Rebuild 04's Laboratory Report resources

`Patient`, `Encounter`, `ServiceRequest`, the panel-level `Observation`, `Binary` +
`DocumentReference`, and `DiagnosticReport` - unchanged from
`04-laboratory-report-fhir-from-hl7v2.ipynb` (see that notebook for the per-field
rationale on each), **except** `Patient.text` - one addition this notebook makes on top
of 04, explained below - plus a new `Specimen` resource, added below too. Not
re-validated individually here since 04 already validated the unchanged parts; this
notebook only adds new checks where the content actually changes or is genuinely new.

**Why `Patient.text` is new:** section 9's renderer follows
`build.fhir.org/documents.html#presentation` literally - `Composition.subject`'s
narrative comes from *the referenced resource's own* `.text`, not anything built for the
document. 04's `Patient` never had one (nothing in that notebook's own output needed it),
so the "patient demographics" part of a rendered document was silently empty - not a
renderer bug, a gap in the resource the renderer was faithfully reporting. Fixed at the
source here rather than special-cased in the renderer.

In [3]:
patient_fullurl = f"urn:uuid:{uuid4()}"
patient_sex_display = {"M": "Male", "F": "Female"}.get(report_row["sex"], "Unknown")
patient = {
    "resourceType": "Patient",
    "text": {
        "status": "generated",
        "div": (
            '<div xmlns="http://www.w3.org/1999/xhtml">'
            f"<p><b>{report_row['given_name']} {report_row['family_name']}</b></p>"
            f"<p>NHS number: {report_row['nhs_number']}</p>"
            f"<p>Date of birth: {report_row['birth_date']}</p>"
            f"<p>Sex: {patient_sex_display}</p>"
            f"<p>Postcode: {report_row['postcode']}</p>"
            "</div>"
        ),
    },
    "identifier": [
        {"system": NHS_NUMBER_SYSTEM, "type": {"coding": [{"system": V2_0203, "code": "NH"}]}, "value": report_row["nhs_number"]},
        {"assigner": {"identifier": {"system": ODS_SYSTEM, "value": report_row["mrn_assigner_ods"]}},
         "type": {"coding": [{"system": V2_0203, "code": "MR"}]}, "value": report_row["mrn"]},
    ],
    "name": [{"family": report_row["family_name"], "given": [report_row["given_name"]]}],
    "gender": {"M": "male", "F": "female"}.get(report_row["sex"], "unknown"),
    "birthDate": report_row["birth_date"],
    "address": [{"postalCode": report_row["postcode"]}],
}

#### Specimen - the sample the test was run on

The source message has no `SPM` segment (this fixture predates NW-GMSA's own convention
of including one) and `OBR-15` (Specimen Source) is blank, so there's no free-text
specimen type to map the way `03-laboratory-order-from-csv.ipynb` did from its CSV's
`SpecimenTypeDescription`. `OBR-14` (Specimen Received Date/Time) is the one specimen
fact this message actually carries, so that's the one field taken from it; the specimen
`type` reuses the same SNOMED code 03 used for its own blood specimen -
[`258580003` "Whole blood specimen"](https://nw-gmsa.github.io/en/StructureDefinition-Specimen.html) -
appropriate here too, since a ctDNA panel like this one is run on a blood draw. No
`collection.collectedDateTime` or specimen accession identifier: neither is in the
source, so neither is invented. Built ahead of `ServiceRequest`/`DiagnosticReport` below
since both reference it (`.specimen`).

In [4]:
specimen_fullurl = f"urn:uuid:{uuid4()}"
specimen = {
    "resourceType": "Specimen",
    "meta": {"profile": [SPECIMEN_PROFILE]},
    "status": "available",
    "type": {"coding": [{"system": "http://snomed.info/sct", "code": "258580003", "display": "Whole blood specimen"}]},
    "subject": {"reference": patient_fullurl,
                "identifier": {"system": NHS_NUMBER_SYSTEM, "type": {"coding": [{"system": V2_0203, "code": "NH"}]}, "value": report_row["nhs_number"]}},
    "receivedTime": report_row["specimen_received_datetime"],
}

validate_resource(specimen, ["package.tgz"], [SPECIMEN_PROFILE])

,severity,code,details,expression


In [5]:
encounter_fullurl = f"urn:uuid:{uuid4()}"
encounter = {
    "resourceType": "Encounter",
    "status": "finished",
    "class": {"system": "http://terminology.hl7.org/CodeSystem/v3-ActCode", "code": "OBSENC"},
    "identifier": [{"type": {"coding": [{"system": V2_0203, "code": "AN"}]}, "value": report_row["account_number"]}],
    "subject": {"reference": patient_fullurl,
                "identifier": {"system": NHS_NUMBER_SYSTEM, "type": {"coding": [{"system": V2_0203, "code": "NH"}]}, "value": report_row["nhs_number"]}},
}

service_request_fullurl = f"urn:uuid:{uuid4()}"
service_request = {
    "resourceType": "ServiceRequest",
    "status": "completed",
    "intent": "order",
    "category": [{"coding": [{"system": "http://snomed.info/sct", "code": "116148004"}]}],
    "code": {"coding": [{"system": GENOMIC_TEST_DIRECTORY_SYSTEM, "code": report_row["test_directory_code"]}]},
    "identifier": [
        {"assigner": {"identifier": {"system": ODS_SYSTEM, "value": report_row["ordering_org_ods"]}},
         "type": {"coding": [{"system": V2_0203, "code": "PLAC"}]}, "value": report_row["placer_order_number"]},
        {"assigner": {"identifier": {"system": ODS_SYSTEM, "value": GLH_ODS}}, "system": IGENE_REPORT_ID_SYSTEM,
         "type": {"coding": [{"system": V2_0203, "code": "FILL"}]}, "value": report_row["filler_report_number"]},
    ],
    "subject": {"reference": patient_fullurl,
                "identifier": {"system": NHS_NUMBER_SYSTEM, "type": {"coding": [{"system": V2_0203, "code": "NH"}]}, "value": report_row["nhs_number"]}},
    "requester": {"display": report_row["ordering_org_name"],
                  "identifier": {"system": ODS_SYSTEM, "value": report_row["ordering_org_ods"]}, "type": "Organization"},
    "reasonCode": [{"coding": [{"system": GENOMIC_CLINICAL_INDICATION_SYSTEM, "code": report_row["clinical_indication_code"]}]}],
    "encounter": {"reference": encounter_fullurl, "identifier": {"type": {"coding": [{"system": V2_0203, "code": "AN"}]}, "value": report_row["account_number"]}},
    "specimen": [{"reference": specimen_fullurl, "type": "Specimen"}],
}

In [6]:
panel_observation_fullurl = f"urn:uuid:{uuid4()}"
panel_observation = {
    "resourceType": "Observation",
    "meta": {"profile": ["https://fhir.nwgenomics.nhs.uk/StructureDefinition/GenomicStudyPanel"]},
    "status": "final",
    "category": [
        {"coding": [{"system": "http://terminology.hl7.org/CodeSystem/observation-category", "code": "laboratory"}]},
        {"coding": [{"system": "http://terminology.hl7.org/CodeSystem/v2-0074", "code": "GE"}]},
    ],
    "code": {"coding": [{"system": "http://loinc.org", "code": "81306-3", "display": "Variables that apply to the overall study"}]},
    "identifier": [
        {"assigner": {"identifier": {"system": ODS_SYSTEM, "value": GLH_ODS}}, "system": IGENE_REPORT_ID_SYSTEM,
         "type": {"coding": [{"system": V2_0203, "code": "FILL"}]}, "value": report_row["filler_report_number"]}
    ],
    "subject": {"reference": patient_fullurl,
                "identifier": {"system": NHS_NUMBER_SYSTEM, "type": {"coding": [{"system": V2_0203, "code": "NH"}]}, "value": report_row["nhs_number"]}},
    "effectiveDateTime": report_row["report_datetime"],
    "component": [
        {"code": {"coding": [{"system": "http://loinc.org", "code": "51967-8"}]},
         "valueCodeableConcept": {"coding": [{"system": GENOMIC_CLINICAL_INDICATION_SYSTEM, "code": report_row["clinical_indication_code"]}]}},
        {"code": {"coding": [{"system": "http://loinc.org", "code": "51968-6"}]},
         "valueCodeableConcept": {"coding": [{"system": "https://fhir.nwgenomics.nhs.uk/CodeSystem/GenomicTestOutcomeCode",
                                               "code": report_row["outcome_code"], "display": report_row["outcome_display"]}]}},
    ],
}

binary_fullurl = f"urn:uuid:{uuid4()}"
binary = {"resourceType": "Binary", "contentType": report_row["pdf_content_type"], "data": report_row["pdf_base64"]}

document_reference_fullurl = f"urn:uuid:{uuid4()}"
document_reference = {
    "resourceType": "DocumentReference",
    "status": "current",
    "type": {"coding": [{"system": "http://snomed.info/sct", "code": "1054161000000101", "display": "Genetic report"}]},
    "subject": {"reference": patient_fullurl,
                "identifier": {"system": NHS_NUMBER_SYSTEM, "type": {"coding": [{"system": V2_0203, "code": "NH"}]}, "value": report_row["nhs_number"]}},
    "date": report_row["report_datetime"],
    "identifier": [
        {"assigner": {"identifier": {"system": ODS_SYSTEM, "value": GLH_ODS}}, "system": IGENE_REPORT_ID_SYSTEM,
         "type": {"coding": [{"system": V2_0203, "code": "FILL"}]}, "value": report_row["filler_report_number"]}
    ],
    "custodian": {"identifier": {"system": ODS_SYSTEM, "value": GLH_ODS}, "type": "Organization"},
    "content": [{"attachment": {"contentType": report_row["pdf_content_type"], "url": binary_fullurl}}],
    "context": {
        "period": {"start": report_row["report_datetime"], "end": report_row["report_datetime"]},
        "encounter": [{"reference": encounter_fullurl, "type": "Encounter",
                       "identifier": {"type": {"coding": [{"system": V2_0203, "code": "AN"}]}, "value": report_row["account_number"]}}],
        "sourcePatientInfo": {"identifier": {"assigner": {"identifier": {"system": ODS_SYSTEM, "value": report_row["mrn_assigner_ods"]}},
                                              "type": {"coding": [{"system": V2_0203, "code": "MR"}]}, "value": report_row["mrn"]}},
        "related": [{"reference": service_request_fullurl, "type": "ServiceRequest",
                     "identifier": {"assigner": {"identifier": {"system": ODS_SYSTEM, "value": report_row["ordering_org_ods"]}},
                                    "type": {"coding": [{"system": V2_0203, "code": "PLAC"}]}, "value": report_row["placer_order_number"]}}],
    },
}

In [7]:
diagnostic_report_fullurl = f"urn:uuid:{uuid4()}"
diagnostic_report = {
    "resourceType": "DiagnosticReport",
    "status": {"F": "final"}.get(report_row["result_status"], "unknown"),
    "category": [{"coding": [{"system": "http://terminology.hl7.org/CodeSystem/v2-0074", "code": "GE"}]}],
    "code": {
        "coding": [
            {"system": IGEAP_SYSTEM, "code": report_row["test_code_local"], "display": report_row["test_description"]},
            {"system": GENOMIC_TEST_DIRECTORY_SYSTEM, "code": report_row["test_directory_code"], "display": report_row["test_description"]},
        ]
    },
    "subject": {"reference": patient_fullurl,
                "identifier": {"system": NHS_NUMBER_SYSTEM, "type": {"coding": [{"system": V2_0203, "code": "NH"}]}, "value": report_row["nhs_number"]}},
    "encounter": {"reference": encounter_fullurl, "identifier": {"type": {"coding": [{"system": V2_0203, "code": "AN"}]}, "value": report_row["account_number"]}},
    "effectiveDateTime": report_row["report_datetime"],
    "issued": report_row["report_datetime"],
    "identifier": [
        {"assigner": {"identifier": {"system": ODS_SYSTEM, "value": GLH_ODS}}, "system": IGENE_REPORT_ID_SYSTEM,
         "type": {"coding": [{"system": V2_0203, "code": "FILL"}]}, "value": report_row["filler_report_number"]}
    ],
    "basedOn": [{"reference": service_request_fullurl, "type": "ServiceRequest",
                 "identifier": {"assigner": {"identifier": {"system": ODS_SYSTEM, "value": report_row["ordering_org_ods"]}},
                                "type": {"coding": [{"system": V2_0203, "code": "PLAC"}]}, "value": report_row["placer_order_number"]}}],
    "result": [{"reference": panel_observation_fullurl, "type": "Observation", "display": "Variables that apply to the overall study"}],
    "specimen": [{"reference": specimen_fullurl, "type": "Specimen"}],
    "resultsInterpreter": [{"display": f"{report_row['interpreter_given']} {report_row['interpreter_family']}"}],
    "performer": [{"display": GLH_NAME, "identifier": {"system": ODS_SYSTEM, "value": GLH_ODS}, "type": "Organization"}],
    "conclusionCode": [{"coding": [{"system": "https://fhir.nwgenomics.nhs.uk/CodeSystem/GenomicTestOutcomeCode",
                                     "code": report_row["outcome_code"], "display": report_row["outcome_display"]}]}],
    "presentedForm": [{"contentType": report_row["pdf_content_type"], "url": binary_fullurl}],
}

### 3. Rebuild 05's variant `Observation`s — this time, resolving identity from the start

`05-test-results-from-vcf.ipynb` had to build its *own* `Patient`/`Organization` because
it was run standalone, with no Laboratory Report `Bundle` already in memory to borrow
from. Its `build_observation()` takes `patient_ref` as a plain parameter for exactly this
reason (see its own section 17: *"specifically so this notebook can reuse it against a
different (real order) context"*) — so here, merging is simply a matter of calling it
with `patient_fullurl` from section 2 above instead of building a second `Patient` we'd
only have to reconcile away afterwards. The `Organization` (the GLH performer the
Genomics Reporting IG's `variant.performer` needs) has no equivalent already in scope
from section 2, so that one small resource is still built fresh — everything else about
Stage 1 (VCF parsing, lookup tables, `build_observation()` itself) is copied unchanged
from 05; see that notebook for the field-by-field mapping rationale.

In [8]:
VCF_PATH = Path("Input/DSS/VCF/igene_example_data.vcf")


def parse_vcf_records(path):
    records = []
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            line = line.rstrip("\n").rstrip("\r")
            if not line or line.startswith("#"):
                continue
            chrom, pos, vid, ref, alt, qual, filt, info, fmt, sample = line.split("\t")
            info_dict = {}
            for item in info.split(";"):
                if "=" in item:
                    k, v = item.split("=", 1)
                    info_dict[k] = v
                else:
                    info_dict[item] = True
            format_dict = dict(zip(fmt.split(":"), sample.split(":")))
            records.append({"chrom": chrom, "pos": int(pos), "id": vid, "ref": ref, "alt": alt, "info": info_dict, "format": format_dict})
    return records


RECORDS = parse_vcf_records(VCF_PATH)

GRCH37_REFSEQ = {"17": "NC_000017.10", "15": "NC_000015.9", "X": "NC_000023.10"}
HGNC_GENE = {"BRCA1": "HGNC:1100", "FBN1": "HGNC:3603"}
SO_TERM = {
    "SNV": ("SO:0001483", "SNV"),
    "deletion": ("SO:0000159", "deletion"),
    "insertion": ("SO:0000667", "insertion"),
    "copy_number_variation": ("SO:0001019", "copy_number_variation"),
}
ALLELIC_STATE = {
    "Heterozygous": ("LA6706-1", "heterozygous"),
    "Homozygous": ("LA6705-3", "homozygous"),
    "Hemizygous": ("LA6707-9", "hemizygous"),
}
INHERITANCE_ORIGIN = {"Maternal": ("LA26320-4", "Maternal")}
len(RECORDS)

4

In [9]:
import re


def cc(system=None, code=None, display=None, text=None):
    concept = {}
    if system and code:
        coding = {"system": system, "code": code}
        if display:
            coding["display"] = display
        concept["coding"] = [coding]
    if text:
        concept["text"] = text
    elif display and "coding" not in concept:
        concept["text"] = display
    return concept


def component(loinc_code, loinc_display, **value):
    comp = {"code": {"coding": [{"system": "http://loinc.org", "code": loinc_code, "display": loinc_display}]}}
    comp.update(value)
    return comp


def dna_change_type(record):
    vartype = record["info"].get("VARTYPE", "")
    if "Copy_Number_Variant" in vartype:
        return SO_TERM["copy_number_variation"]
    if vartype == "Structural_Variant":
        return SO_TERM["deletion"] if record["info"].get("SVTYPE") == "DEL" else None
    ref, alt = record["ref"], record["alt"]
    if len(ref) == 1 and len(alt) == 1:
        return SO_TERM["SNV"]
    if len(alt) < len(ref):
        return SO_TERM["deletion"]
    if len(alt) > len(ref):
        return SO_TERM["insertion"]
    return None


def build_observation(record, obs_id, patient_ref, org_ref, effective_date):
    info = record["info"]
    fmt = record["format"]
    vartype = info.get("VARTYPE", "")
    is_structural = vartype in ("Intragenic_Copy_Number_Variant", "Multigenic_Copy_Number_Variant", "Structural_Variant")

    components = []

    gene = info.get("GENE")
    if gene:
        hgnc = HGNC_GENE.get(gene)
        components.append(component("48018-6", "Gene studied [ID]",
                                     valueCodeableConcept=cc("http://www.genenames.org", hgnc, gene) if hgnc else cc(text=gene)))

    if "INHERITANCE" in info:
        components.append(component("48002-0", "Genomic source class [Type]",
                                     valueCodeableConcept=cc("http://loinc.org", "LA6683-2", "Germline")))

    refseq = GRCH37_REFSEQ.get(record["chrom"])
    if refseq:
        components.append(component("48013-7", "Genomic reference sequence [ID]",
                                     valueCodeableConcept=cc("http://www.ncbi.nlm.nih.gov/refseq", refseq)))

    components.append(component("92822-6", "Genomic coordinate system [Type]",
                                 valueCodeableConcept=cc("http://loinc.org", "LA30102-0", "1-based character counting")))
    components.append(component("69547-8", "Genomic ref allele [ID]", valueString=record["ref"]))
    components.append(component("69551-0", "Genomic alt allele [ID]", valueString=record["alt"]))

    so_term = dna_change_type(record)
    if so_term:
        code_val, display = so_term
        components.append(component("48019-4", "DNA change type",
                                     valueCodeableConcept=cc("http://www.sequenceontology.org", code_val, display)))

    hgvsc = info.get("HGVSC")
    if hgvsc:
        transcript_match = re.match(r"(N[MR]_\d+\.\d+)", hgvsc)
        if transcript_match:
            components.append(component("51958-7", "Transcript reference sequence [ID]",
                                         valueCodeableConcept=cc("http://www.ncbi.nlm.nih.gov/refseq", transcript_match.group(1))))
        components.append(component("48004-6", "DNA change (c.HGVS)", valueCodeableConcept=cc("http://varnomen.hgvs.org", hgvsc)))

    hgvsp = info.get("HGVSP")
    if hgvsp:
        components.append(component("48005-3", "Amino acid change (pHGVS)", valueCodeableConcept=cc("http://varnomen.hgvs.org", hgvsp)))

    hgvsg = info.get("HGVSG")
    if hgvsg:
        components.append(component("81290-9", "Genomic DNA change (gHGVS)", valueCodeableConcept=cc("http://varnomen.hgvs.org", hgvsg)))

    cytoband = info.get("CYTOBAND")
    if cytoband:
        components.append(component("48001-2", "Cytogenetic (chromosome) location", valueCodeableConcept=cc(text=cytoband)))

    classification = info.get("CLASS")
    if classification:
        components.append(component("53037-8", "Genetic variation clinical significance [Imp]",
                                     valueCodeableConcept=cc(text=classification.replace("_", " "))))

    inheritance = info.get("INHERITANCE")
    if inheritance:
        origin = INHERITANCE_ORIGIN.get(inheritance)
        components.append(component("94186-4", "Origin of germline genetic variant [Type]",
                                     valueCodeableConcept=cc("http://loinc.org", *origin) if origin else cc(text=inheritance)))

    end = info.get("END")
    if is_structural and end:
        components.append(component("81302-2", "Structural variant inner start and end",
                                     valueRange={"low": {"value": record["pos"]}, "high": {"value": int(end)}}))
    elif not is_structural:
        components.append(component("81254-5", "Genomic allele start-end", valueRange={"low": {"value": record["pos"]}}))

    vaf = fmt.get("VAF")
    if vaf and vaf != ".":
        components.append(component("81258-6", "Sample variant allelic frequency",
                                     valueQuantity={"value": float(vaf), "unit": "decimal", "system": "http://unitsofmeasure.org"}))

    zyg = fmt.get("ZYG")
    if zyg:
        copy_match = re.search(r"\((\d+)_cop(?:y|ies)\)", zyg)
        if copy_match:
            components.append(component("82155-3", "Genomic structural variant copy number",
                                         valueQuantity={"value": int(copy_match.group(1)), "system": "http://unitsofmeasure.org", "code": "1"}))
        elif zyg in ALLELIC_STATE:
            components.append(component("53034-5", "Allelic state", valueCodeableConcept=cc("http://loinc.org", *ALLELIC_STATE[zyg])))

    return {
        "resourceType": "Observation",
        "id": obs_id,
        "meta": {"profile": [GENOMICS_VARIANT_PROFILE]},
        "status": "final",
        "category": [
            {"coding": [{"system": "http://terminology.hl7.org/CodeSystem/observation-category", "code": "laboratory"}]},
            {"coding": [{"system": "http://terminology.hl7.org/CodeSystem/v2-0074", "code": "GE"}]},
        ],
        "code": {"coding": [{"system": "http://loinc.org", "code": "69548-6", "display": "Genetic variant assessment"}]},
        "subject": {"reference": patient_ref},
        "effectiveDateTime": effective_date,
        "performer": [{"reference": org_ref}],
        "valueCodeableConcept": cc("http://loinc.org", "LA9633-4", "Present"),
        "method": cc("http://loinc.org", "LA26398-0", "Sequencing"),
        "component": components,
    }

In [10]:
organization_fullurl = f"urn:uuid:{uuid4()}"
organization = {
    "resourceType": "Organization",
    "identifier": [{"system": ODS_SYSTEM, "value": GLH_ODS}],
    "name": GLH_NAME,
}

variant_observations = [
    # A resource `.id` referenced via a `urn:uuid:` fullUrl must itself be a valid UUID -
    # 05's own placeholder-style ids ("ctdna...-seqv1") aren't, which the validator only
    # flags once these Observations sit inside a whole-Bundle check (section 4/7 below),
    # not during 05's own single-resource validation. Use a real uuid4() here instead.
    build_observation(record, str(uuid4()), patient_fullurl, organization_fullurl, report_row["report_datetime"])
    for record in RECORDS
]
len(variant_observations)

4

### 4. Stack the NW-GMSA `Observation` profile on top too

`DiagnosticReport.result` (section 5) is about to point at these `Observation`s from a
NW-GMSA `DiagnosticReport` alongside the existing panel `Observation` — and NW-GMSA's own
`DiagnosticReport` profile constrains `result` to target its *own* `Observation` profile,
which the panel `Observation` already satisfies (it's built as a `GenomicStudyPanel`, a
specialisation of it) but the Genomics-Reporting-only variant `Observation`s don't yet.
Same fix as section 5 uses for the `DiagnosticReport` itself — "a profile stacks
requirements rather than replacing them" (05's own words, section 21) — add
`NWGMSA_OBSERVATION_PROFILE` to `meta.profile` alongside `GENOMICS_VARIANT_PROFILE`. That
profile's one extra requirement neither `build_observation()` nor the Genomics Reporting
IG already provides is `Observation.identifier`, so each variant gets one, built the same
way as every other identifier in this series (`IGENE_REPORT_ID_SYSTEM`, assigned by the
GLH, distinguished per variant by a `-N` suffix on the report's own filler number).

In [11]:
for i, obs in enumerate(variant_observations, start=1):
    obs["meta"]["profile"].append(NWGMSA_OBSERVATION_PROFILE)
    obs["identifier"] = [{
        "assigner": {"identifier": {"system": ODS_SYSTEM, "value": GLH_ODS}},
        "system": IGENE_REPORT_ID_SYSTEM,
        "type": {"coding": [{"system": V2_0203, "code": "FILL"}]},
        "value": f"{report_row['filler_report_number']}-{i}",
    }]

### 5. Validate the variant Observations

Against the real `Patient` this time, not a placeholder — unlike 05's Stage 1 (which
proved the shape in isolation first, since it had no real order context yet at that
point), this notebook already has the real one in scope from section 2. Both profiles
checked at once, same as every dual-conformant resource in this notebook.

In [12]:
rows = []
for obs in variant_observations:
    df = validate_resource(obs, ["package.tgz", GENOMICS_REPORTING_IG], [NWGMSA_OBSERVATION_PROFILE, GENOMICS_VARIANT_PROFILE])
    df.insert(0, "observation", obs["id"])
    rows.append(df)

pd.concat(rows, ignore_index=True)

,observation,severity,code,details,expression
0,f866eb21-c93b-4ea6-8e79-780149b39c6e,information,informational,This element does not match any known slice de...,[Observation.component[0].value.ofType(Codeabl...
1,f866eb21-c93b-4ea6-8e79-780149b39c6e,information,code-invalid,Binding for path Observation.component[7].valu...,[Observation.component[7].value.ofType(Codeabl...
2,f866eb21-c93b-4ea6-8e79-780149b39c6e,information,code-invalid,Binding for path Observation.component[2].valu...,[Observation.component[2].value.ofType(Codeabl...
3,f866eb21-c93b-4ea6-8e79-780149b39c6e,information,informational,This element does not match any known slice de...,[Observation.component[11]]
4,f866eb21-c93b-4ea6-8e79-780149b39c6e,information,informational,This element does not match any known slice de...,[Observation.component[15].value.ofType(Codeab...
...,...,...,...,...,...
86,7f70a970-2deb-47c6-9b4d-a252bda6bdd6,warning,not-found,A definition for CodeSystem 'http://varnomen.h...,[Observation.component[5].value.ofType(Codeabl...
87,7f70a970-2deb-47c6-9b4d-a252bda6bdd6,warning,not-found,A definition for CodeSystem 'http://www.sequen...,[Observation.component[4].value.ofType(Codeabl...
88,7f70a970-2deb-47c6-9b4d-a252bda6bdd6,warning,not-found,A definition for CodeSystem 'http://www.sequen...,[Observation.component[4].value.ofType(Codeabl...
89,7f70a970-2deb-47c6-9b4d-a252bda6bdd6,warning,not-found,A definition for CodeSystem 'http://www.ncbi.n...,[Observation.component[0].value.ofType(Codeabl...


### 6. Merge into one `DiagnosticReport`

The Laboratory Report's own `DiagnosticReport` (section 2) already has one `result` —
the panel-level `Observation` — so the four variant `Observation`s are *appended*, not
used to replace it. `meta.profile` gains the Genomics Reporting IG's `genomic-report`
profile alongside NW-GMSA's own, and `code.coding` gains that profile's required fixed
LOINC `51969-4` "Genetic analysis report" coding — the same two additions 05 made to its
own (now-discarded) copy of this `DiagnosticReport`, section 21. Everything else about
the resource is untouched.

The Test Results message's own `DiagnosticReport` and `Patient`/`Encounter`/
`ServiceRequest` are not carried into this notebook at all — reusing `patient_fullurl`
directly in section 3 meant there was never a duplicate copy of any of them to discard in
the first place.

In [13]:
merged_diagnostic_report = copy.deepcopy(diagnostic_report)
merged_diagnostic_report["meta"] = {"profile": [DIAGNOSTIC_REPORT_PROFILE, GENOMICS_REPORT_PROFILE]}
merged_diagnostic_report["code"]["coding"].append({"system": "http://loinc.org", "code": "51969-4", "display": "Genetic analysis report"})
merged_diagnostic_report["result"] += [
    {"reference": f"urn:uuid:{obs['id']}", "type": "Observation"} for obs in variant_observations
]

validate_resource(merged_diagnostic_report, ["package.tgz", GENOMICS_REPORTING_IG], [DIAGNOSTIC_REPORT_PROFILE, GENOMICS_REPORT_PROFILE])

,severity,code,details,expression
3,error,structure,Slice 'DiagnosticReport.code.coding:UKCoreRepo...,[DiagnosticReport.code]
5,error,processing,Slicing cannot be evaluated: Could not match d...,[DiagnosticReport.conclusionCode[0].coding[0]]
0,information,informational,This element does not match any known slice de...,[DiagnosticReport.resultsInterpreter[0]]
1,information,informational,This element does not match any known slice de...,[DiagnosticReport.code.coding[0]]
2,information,informational,This element does not match any known slice de...,[DiagnosticReport.code.coding[2]]
4,information,business-rule,Reference to draft CodeSystem https://fhir.nwg...,[DiagnosticReport.conclusionCode[0]]
6,information,not-supported,DiagnosticReport.conclusionCode.coding:Genomic...,[DiagnosticReport.conclusionCode[0]]
10,information,informational,This element does not match any known slice de...,[DiagnosticReport.result[0]]
11,information,informational,This element does not match any known slice de...,[DiagnosticReport.result[1]]
12,information,informational,This element does not match any known slice de...,[DiagnosticReport.result[2]]


### 7. Build the `Composition`

Modelled directly on the EU Laboratory IG's own published example,
`Composition-comp-lab-example.json` — `type`/`category`, the
`composition-diagnosticReportReference` extension, and the section shape (`title`/
`code`/`text`/`entry`) are all copied from its structure rather than re-derived from the
profile alone. Three sections, each demonstrating the "narrative and structured at once"
point from section 0:

- **Laboratory Report** — coded with LOINC `81306-3` (the same code already used as the
  panel `Observation`'s own `code` in section 2 — this section *is* that observation's
  narrative, in prose). Its text now names both order numbers (`ORC-2`/`ORC-3` in the
  original v2 message, `report_row["placer_order_number"]`/`["filler_report_number"]`
  here) alongside the test itself — identifiers that were already on `ServiceRequest`/
  `DiagnosticReport` (section 2) but hadn't made it into anything a reader actually sees.
  `entry` points at the merged `DiagnosticReport`.
- **Specimen** — coded with SNOMED `123038009` "Specimen" (the generic top-level SNOMED
  concept for a specimen, since there's no more specific LOINC/SNOMED code this
  particular section is *about* beyond "the specimen"), narrative built straight from the
  `Specimen` resource (section 2). `entry` points at it.
- **Genomic Findings** — coded with LOINC `69548-6` (the variant `Observation`s' own
  `code`, section 3). Its narrative is an HTML table built by reading straight out of
  each variant `Observation`'s own `component[]` (`component_value()`, section 2's helper
  cell) — the table's numbers are the resource's numbers, not a separately-maintained
  copy of them. One `entry` per variant `Observation`.

In [14]:
def variant_row_html(obs):
    gene = component_value(obs, "48018-6") or "-"
    hgvsc = component_value(obs, "48004-6") or "-"
    hgvsp = component_value(obs, "48005-3") or "-"
    change_type = component_value(obs, "48019-4") or "-"
    classification = component_value(obs, "53037-8") or "-"
    zygosity = component_value(obs, "53034-5") or "-"
    return (
        f"<tr><td>{gene}</td><td>{hgvsc}</td><td>{hgvsp}</td>"
        f"<td>{change_type}</td><td>{zygosity}</td><td>{classification}</td></tr>"
    )


laboratory_report_section_html = (
    "<div xmlns=\"http://www.w3.org/1999/xhtml\">"
    f"<p><b>{report_row['test_description']}</b> ({report_row['test_directory_code']}), "
    f"requested by {report_row['ordering_org_name']}.</p>"
    f"<p>Placer order number: {report_row['placer_order_number']}. "
    f"Filler order number: {report_row['filler_report_number']}.</p>"
    f"<p>Result: <b>{report_row['outcome_display']}</b>. "
    f"Reported by {GLH_NAME}, interpreted by {report_row['interpreter_given']} {report_row['interpreter_family']}.</p>"
    "</div>"
)

specimen_section_html = (
    "<div xmlns=\"http://www.w3.org/1999/xhtml\">"
    f"<p><b>{specimen['type']['coding'][0]['display']}</b></p>"
    f"<p>Received: {specimen['receivedTime']}</p>"
    "</div>"
)

genomic_findings_section_html = (
    "<div xmlns=\"http://www.w3.org/1999/xhtml\">"
    "<table class=\"grid\"><tr><th>Gene</th><th>DNA change (c.HGVS)</th><th>Protein change (p.HGVS)</th>"
    "<th>Change type</th><th>Zygosity</th><th>Classification</th></tr>"
    + "".join(variant_row_html(obs) for obs in variant_observations)
    + "</table></div>"
)

print(genomic_findings_section_html)

<div xmlns="http://www.w3.org/1999/xhtml"><table class="grid"><tr><th>Gene</th><th>DNA change (c.HGVS)</th><th>Protein change (p.HGVS)</th><th>Change type</th><th>Zygosity</th><th>Classification</th></tr><tr><td>BRCA1</td><td>NM_007294.3(BRCA1):c.68_69del</td><td>p.(Glu23ValfsTer17)</td><td>deletion</td><td>heterozygous</td><td>Pathogenic</td></tr><tr><td>FBN1</td><td>NM_000138.4(FBN1):exon13_to_exon15del</td><td>-</td><td>copy_number_variation</td><td>-</td><td>Pathogenic</td></tr><tr><td>-</td><td>-</td><td>-</td><td>copy_number_variation</td><td>-</td><td>Pathogenic</td></tr><tr><td>-</td><td>-</td><td>-</td><td>deletion</td><td>-</td><td>Pathogenic</td></tr></table></div>


In [15]:
genomics_composition_fullurl = f"urn:uuid:{uuid4()}"
genomics_composition = {
    "resourceType": "Composition",
    "meta": {"profile": [COMPOSITION_EU_LAB_PROFILE]},
    "extension": [{"url": DIAGNOSTIC_REPORT_REFERENCE_EXT, "valueReference": {"reference": diagnostic_report_fullurl}}],
    "identifier": {"system": "urn:ietf:rfc:3986", "value": f"urn:uuid:{uuid4()}"},
    "status": "final",
    "type": {"coding": [{"system": "http://loinc.org", "code": "11502-2", "display": "Laboratory report"}]},
    "category": [{"coding": [{"system": "http://loinc.org", "code": "26436-6", "display": "Laboratory Studies (set)"}]}],
    "subject": {"reference": patient_fullurl,
                "identifier": {"system": NHS_NUMBER_SYSTEM, "type": {"coding": [{"system": V2_0203, "code": "NH"}]}, "value": report_row["nhs_number"]}},
    "date": report_row["report_datetime"],
    "author": [{"display": GLH_NAME, "identifier": {"system": ODS_SYSTEM, "value": GLH_ODS}, "type": "Organization"}],
    "title": f"{report_row['test_description']} - Laboratory Report and Genomic Findings",
    "section": [
        {
            "title": "Laboratory Report",
            "code": {"coding": [{"system": "http://loinc.org", "code": "81306-3", "display": "Variables that apply to the overall study"}]},
            "text": {"status": "generated", "div": laboratory_report_section_html},
            "entry": [{"reference": diagnostic_report_fullurl, "type": "DiagnosticReport"}],
        },
        {
            "title": "Specimen",
            "code": {"coding": [{"system": "http://snomed.info/sct", "code": "123038009", "display": "Specimen"}]},
            "text": {"status": "generated", "div": specimen_section_html},
            "entry": [{"reference": specimen_fullurl, "type": "Specimen"}],
        },
        {
            "title": "Genomic Findings",
            "code": {"coding": [{"system": "http://loinc.org", "code": "69548-6", "display": "Genetic variant assessment"}]},
            "text": {"status": "generated", "div": genomic_findings_section_html},
            "entry": [{"reference": f"urn:uuid:{obs['id']}", "type": "Observation"} for obs in variant_observations],
        },
    ],
}

validate_resource(genomics_composition, [EU_LAB_IG], [COMPOSITION_EU_LAB_PROFILE])

,severity,code,details,expression
0,information,informational,This element does not match any known slice de...,[Composition.category[0]]
1,information,informational,This element does not match any known slice de...,[Composition.section[0]]
2,information,informational,This element does not match any known slice de...,[Composition.section[1]]
3,information,informational,This element does not match any known slice de...,[Composition.section[2]]
4,warning,code-invalid,None of the codings provided are in the value ...,[Composition.subject.identifier.type]


### 8. Assemble the Document `Bundle`

`Bundle.type = "document"`, `Composition` as `entry[0]` (required — a Document `Bundle`
is defined by its first entry being the `Composition`, everything else is that
Composition's supporting content). Every resource either the `Composition` or the merged
`DiagnosticReport` references follows: `Patient`, `Specimen`, `Encounter`,
`ServiceRequest`, `Organization` (needed only because the variant `Observation`s'
`performer` points at it), `Binary` + `DocumentReference` (the PDF), the merged
`DiagnosticReport`, the panel `Observation`, and the four variant `Observation`s.

Whole-bundle validation runs all three IGs from section 0 in one call — `package.tgz` for
the NW-GMSA `DiagnosticReport` shape, `hl7.fhir.eu.laboratory#2.0.0` for the `Composition`
and `Bundle` themselves, and the Genomics Reporting IG for the `genomic-report`/`variant`
profiles — the same repeatable-`-bundle`-flags approach 05 used for its two-IG check.

In [16]:
genomics_document_bundle = {
    "resourceType": "Bundle",
    # bdl-9: a document Bundle's identifier needs both .system and .value, unlike the
    # .value-only identifier 04/05's message Bundles used (bdl-9 only applies to documents).
    "identifier": {"system": "urn:ietf:rfc:3986", "value": f"urn:uuid:{uuid4()}"},
    "timestamp": datetime.now().astimezone().strftime("%Y-%m-%dT%H:%M:%S+00:00"),
    "type": "document",
    "entry": (
        [
            {"fullUrl": genomics_composition_fullurl, "resource": genomics_composition},
            {"fullUrl": patient_fullurl, "resource": patient},
            {"fullUrl": specimen_fullurl, "resource": specimen},
            {"fullUrl": encounter_fullurl, "resource": encounter},
            {"fullUrl": service_request_fullurl, "resource": service_request},
            {"fullUrl": organization_fullurl, "resource": organization},
            {"fullUrl": binary_fullurl, "resource": binary},
            {"fullUrl": document_reference_fullurl, "resource": document_reference},
            {"fullUrl": diagnostic_report_fullurl, "resource": merged_diagnostic_report},
            {"fullUrl": panel_observation_fullurl, "resource": panel_observation},
        ]
        + [{"fullUrl": f"urn:uuid:{obs['id']}", "resource": obs} for obs in variant_observations]
    ),
}

genomics_document_filename = "ctdna9737383222-eulab-document.json"
with open("Input/FHIR/R01/" + genomics_document_filename, "w") as f:
    json.dump(genomics_document_bundle, f, indent=2)

print("Saved Input/FHIR/R01/" + genomics_document_filename)

Saved Input/FHIR/R01/ctdna9737383222-eulab-document.json


In [17]:
genomics_outcome_path = Path("Results/FHIR/R01/" + genomics_document_filename + "-OperationOutcome.json")
composition_index = entry_index(genomics_document_bundle, genomics_composition)
diagnostic_report_index = entry_index(genomics_document_bundle, merged_diagnostic_report)
specimen_index = entry_index(genomics_document_bundle, specimen)

validate_bundle(
    "Input/FHIR/R01/" + genomics_document_filename,
    ["package.tgz", EU_LAB_IG, GENOMICS_REPORTING_IG],
    [
        (f"Composition:{composition_index}", COMPOSITION_EU_LAB_PROFILE),
        (f"DiagnosticReport:{diagnostic_report_index}", DIAGNOSTIC_REPORT_PROFILE),
        (f"DiagnosticReport:{diagnostic_report_index}", GENOMICS_REPORT_PROFILE),
        (f"Specimen:{specimen_index}", SPECIMEN_PROFILE),
    ],
    genomics_outcome_path,
)

,severity,code,details,expression
40,error,processing,Slicing cannot be evaluated: Could not match d...,[Bundle.entry[8].resource/*DiagnosticReport/nu...
118,error,structure,Invalid Resource target type. Found Organizati...,[Bundle.entry[11].resource/*Observation/2728fa...
38,error,structure,Unable to find a profile match for urn:uuid:7f...,[Bundle.entry[8].resource/*DiagnosticReport/nu...
36,error,structure,Unable to find a profile match for urn:uuid:fa...,[Bundle.entry[8].resource/*DiagnosticReport/nu...
34,error,structure,Unable to find a profile match for urn:uuid:27...,[Bundle.entry[8].resource/*DiagnosticReport/nu...
...,...,...,...,...
99,warning,not-found,A definition for CodeSystem 'http://www.ncbi.n...,[Bundle.entry[11].resource/*Observation/2728fa...
98,warning,not-found,A definition for CodeSystem 'http://www.genena...,[Bundle.entry[11].resource/*Observation/2728fa...
144,warning,not-found,A definition for CodeSystem 'http://www.sequen...,[Bundle.entry[12].resource/*Observation/faf18f...
145,warning,not-found,A definition for CodeSystem 'http://varnomen.h...,[Bundle.entry[12].resource/*Observation/faf18f...


A handful of `error`-severity rows here are all confirmed, by isolating and re-checking
the individual resource on its own, to be either pre-existing or specific to whole-Bundle
reference resolution rather than anything wrong with how this notebook builds a resource:

- `Slice 'DiagnosticReport.code.coding:UKCoreReportCode': a matching slice is required,
  but not found` is a third instance of the same category 04 already flagged twice for
  this exact resource shape (`ServiceRequest.authoredOn`, `conclusionCode` slicing, both
  in 04's sections 6 and 9): the section 2 `diagnostic_report`, validated on its own,
  unmerged, against just `package.tgz`, produces the identical error. A gap in
  `code.coding` this message shape has always had, not something section 6's merge
  introduced.
- `Unable to find a profile match for urn:uuid:... among choices:
  .../StructureDefinition/Observation` (once per variant `Observation`) and
  `Invalid Resource target type. Found Organization, but expected one of
  ([PractitionerRole])` both disappear entirely when the same variant `Observation` is
  validated on its own (section 5's per-resource check) against the exact same IGs and
  profiles — they only appear once `-bundle` resolves the actual references inside the
  whole document. The second one is real and worth naming: the Genomics Reporting IG's
  `variant` profile fixes `Observation.performer` to target `PractitionerRole` only,
  while `build_observation()` (unchanged from 05) points it at a plain `Organization` -
  a genuine mismatch between what NW-GMSA's own resources use for a performing
  organisation and what this international IG expects, not a defect introduced by
  merging the two together.

### 9. Render the Document as HTML - porting the official rendering rules

The whole point of building a `Composition` rather than stopping at a Message `Bundle`
(section 0): its `section[].text` is already valid XHTML, written once in section 7 and
never touched since - so rendering the *structured* `Bundle` back into a *human-readable*
page doesn't need much new logic. HL7 itself publishes a reference renderer,
[`DocumentToHTML.xslt`](https://github.com/HL7/fhir/blob/master/implementations/xmltools/DocumentToHTML.xslt) -
"an instantiation of the rendering process for FHIR documents as defined in the FHIR
specification" (its own header comment) - but it's XSLT 1.0 over the FHIR **XML**
representation, and every `Bundle` in this series is JSON. Converting JSON to XML just to
satisfy an XSLT input format adds a whole extra tool dependency and conversion step for
something that should be straightforward - so rather than run that pipeline, the
stylesheet's own template rules are ported to plain Python operating directly on the JSON
below, keeping this notebook's only dependency the same one it's always had (a Python
`dict`).

The rules that matter, read straight out of the XSLT:

1. `<title>`/`<h1>` = `Composition.title`, falling back to "Untitled Document".
2. `Composition.subject` is rendered as a reference: resolve it against the `Bundle`'s
   own entries and show *that* resource's own `.text.div` - **not** anything built for
   this notebook's `Composition.section`. Since none of `patient`/`encounter`/etc. in this
   series carry their own narrative, this step legitimately renders nothing, same as the
   real stylesheet.
3. Each top-level `Composition.section` becomes a `<div>` with a heading (`<h2>`, one
   level per nesting depth) and its own `text.div` verbatim. `section.entry` - the
   *structured* backing data - is never rendered here at all; it doesn't need to be, since
   `section.text` is already that section's complete narrative.
4. Nested `section.section` is **not** rendered - a real limitation the stylesheet's own
   header comment admits ("Work in progress - nesting levels need work"). Ours doesn't
   use nested sections, so this doesn't cost us anything, but it's worth knowing the
   ported behaviour matches the original's gap rather than "fixing" it.
5. Any `h1`-`h6` inside a narrative gets renumbered by adding the current nesting depth
   (capped at `h6`, falling back to `<p>` beyond that) - so a section's own narrative
   headings drop to the right level under the document's `<h1>` instead of colliding
   with it.

Verified against the genuine article: running these same two `Bundle`s through the real
`DocumentToHTML.xslt` (via `xsltproc`, offline, purely as a one-time check - not part of
this notebook's pipeline) produced output identical to this function's, modulo
whitespace.

Checked directly against the spec too, not just the XSLT (which merely
"provides an instantiation of the rendering process" - its own header comment, not the
normative text itself): [build.fhir.org/documents.html §3.4.1.2 Document
Presentation](https://build.fhir.org/documents.html#presentation) requires narrative be
collated in exactly this order:

> "1. The subject resource Narrative 2. The Composition resource Narrative 3. The
> section.text Narratives ... If the document is presented in a different order from
> that given above, it might not represent the original attested content."

- **section 6** is `render_reference(bundle, composition["subject"], ...)`
- **section 7** is `comp_narrative` (`Composition.text`, if present)
- **section 8** is `sections_html`

- in that order - matching the spec, not just the XSLT's own arrangement of them. The
same page goes on: *"To actually build the combined narrative, simply append all the
narrative `<div>` fragments together"* - exactly what `document_to_html()` does, just in
Python instead of XSLT.

#### Styling it - NHS.UK design tokens, not a bespoke palette

The spec is explicit that a document's stylesheet "SHALL NOT alter the presentation in
such a way that it changes the clinical meaning of the content" and should be embedded
rather than externally referenced ("Relative (internal) references SHOULD be used ...
the viewer may be unable to resolve external content") - so this is pure visual styling
(typography, colour, spacing, a header banner, table borders), inlined in a `<style>`
block, with the narrative content and its order from section 9 completely untouched.

Rather than invent a colour palette, the values below - the link/heading colours, body
text colour, border grey, font stack - are read directly out of
[`nhsuk-frontend`](https://github.com/nhsuk/nhsuk-frontend)'s own published stylesheet
(`nhsuk.min.css`), the design system behind nhs.uk, so a report rendered here looks like
it belongs on an NHS service rather than a generic web page.

In [18]:
NHS_CSS = """
body { font-family: Arial, sans-serif; color: #212b32; background-color: #f0f4f5;
       margin: 0; line-height: 1.5; }
.nhsuk-width-container { max-width: 960px; margin: 0 auto; padding: 0 16px; }
.nhsuk-header-banner { background-color: #005eb8; color: #fff; padding: 24px 0; margin-bottom: 24px; }
.nhsuk-header-banner p { margin: 0 0 4px; font-size: 16px; font-weight: 600; letter-spacing: 0.05em; text-transform: uppercase; }
.nhsuk-header-banner h1 { margin: 0; color: #fff; font-size: 32px; font-weight: 600; line-height: 1.1875; }
main { display: block; }
.nhsuk-panel { background: #fff; border: 1px solid #d8dde0; border-left: 6px solid #005eb8;
               border-radius: 4px; padding: 16px 24px; margin-bottom: 24px; }
.nhsuk-panel--subject { border-left-color: #4c6272; }
.nhsuk-panel h2 { margin-top: 0; color: #212b32; font-size: 27px; font-weight: 600; line-height: 1.22222; }
.nhsuk-panel h3 { color: #212b32; font-size: 22px; font-weight: 600; }
.nhsuk-panel p { margin: 0 0 16px; }
.nhsuk-panel p:last-child { margin-bottom: 0; }
table { border-collapse: collapse; width: 100%; margin-top: 8px; font-size: 16px; }
table caption { text-align: left; font-weight: 600; margin-bottom: 8px; }
th, td { text-align: left; padding: 8px 16px 8px 0; border-bottom: 1px solid #d8dde0; }
th { color: #212b32; font-weight: 600; border-bottom: 2px solid #4c6272; }
a { color: #005eb8; }
a:visited { color: #330072; }
footer { color: #4c6272; font-size: 14px; padding: 24px 0 40px; }
"""


_HEADING_RE = re.compile(r"<h([1-6])([^>]*)>(.*?)</h\1>", re.DOTALL)
_XHTML_XMLNS_RE = re.compile(r'\s+xmlns="http://www\.w3\.org/1999/xhtml"')

DOCUMENT_UNTITLED = "Untitled Document"
SECTION_UNTITLED = "Untitled Section"
NO_HUMAN_DISPLAY = "No human-readable content available"


def get_heading_tag(level):
    return f"h{level}" if level <= 6 else "p"


def remap_headings(fragment, nesting_depth):
    # The XSLT rebuilds every xhtml:* element via local-name(), which drops namespace
    # declarations (they aren\'t part of the @* attribute axis in XPath 1.0) - so the
    # xmlns on our narrative fragments\' outer <div> doesn\'t survive either.
    fragment = _XHTML_XMLNS_RE.sub("", fragment)

    def repl(m):
        old_level = int(m.group(1))
        inner = remap_headings(m.group(3), nesting_depth)
        new_tag = get_heading_tag(old_level + nesting_depth)
        return f"<{new_tag}{m.group(2)}>{inner}</{new_tag}>"

    return _HEADING_RE.sub(repl, fragment)


def resolve_entry(bundle, fullurl):
    return next((e["resource"] for e in bundle["entry"] if e.get("fullUrl") == fullurl), None)


def render_reference(bundle, reference_obj, nesting_depth):
    reference_obj = reference_obj or {}
    ref = reference_obj.get("reference")
    if ref:
        target = resolve_entry(bundle, ref)
        text = target.get("text") if target else None
        return remap_headings(text["div"], nesting_depth) if text else ""
    display = (reference_obj.get("display") or "").strip()
    return f"<p>{display}</p>" if display else f"<p>{NO_HUMAN_DISPLAY}</p>"


def render_section(section, nesting_depth):
    # nhsuk-panel: styling hook only (section 10) - the tag/title/text.div content and
    # order are exactly what section 9's rules produce, untouched.
    tag = get_heading_tag(nesting_depth)
    title = (section.get("title") or "").strip() or SECTION_UNTITLED
    text = section.get("text")
    body = remap_headings(text["div"], nesting_depth + 1) if text else ""
    return f'<div class="nhsuk-panel"><{tag}>{title}</{tag}>{body}</div>'


def document_to_html(bundle):
    composition = bundle["entry"][0]["resource"]
    assert bundle.get("type") == "document", "Bundle.type must be \'document\'"
    assert composition["resourceType"] == "Composition", "entry[0] must be the Composition"

    title = (composition.get("title") or "").strip() or DOCUMENT_UNTITLED

    # Collated in the exact order build.fhir.org/documents.html#presentation requires:
    # 1) subject narrative, 2) Composition narrative, 3) section narratives. Each is
    # wrapped in a styled panel only if it actually rendered something, so a resource
    # with no narrative of its own (like our Patient) produces no empty box - omitting an
    # empty wrapper isn't reordering or hiding attested content, there's none to hide.
    subject_html = render_reference(bundle, composition.get("subject"), nesting_depth=2)
    subject_block = f'<div class="nhsuk-panel nhsuk-panel--subject">{subject_html}</div>' if subject_html.strip() else ""

    comp_text = composition.get("text")
    comp_narrative = remap_headings(comp_text["div"], nesting_depth=2) if comp_text else ""
    comp_block = f'<div class="nhsuk-panel nhsuk-panel--subject">{comp_narrative}</div>' if comp_narrative.strip() else ""

    sections_html = "".join(render_section(s, nesting_depth=2) for s in composition.get("section", []))

    return f"""<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>{title}</title>
<style>{NHS_CSS}</style>
</head>
<body>
<div class="nhsuk-header-banner">
<div class="nhsuk-width-container">
<p>NHS North West Genomics &mdash; FHIR Document</p>
<h1>{title}</h1>
</div>
</div>
<main class="nhsuk-width-container">
{subject_block}
{comp_block}
{sections_html}
<footer>Rendered from a FHIR Document Bundle (Composition {composition.get("id") or composition.get("identifier", {}).get("value", "")}) following the narrative order required by <a href="https://build.fhir.org/documents.html#presentation">build.fhir.org/documents.html &sect;3.4.1.2</a>.</footer>
</main>
</body>
</html>"""


os.makedirs("Output/HTML/R01", exist_ok=True)
genomics_html = document_to_html(genomics_document_bundle)
genomics_html_path = "Output/HTML/R01/" + genomics_document_filename.replace(".json", ".html")
with open(genomics_html_path, "w") as f:
    f.write(genomics_html)
print("Saved " + genomics_html_path)

Saved Output/HTML/R01/ctdna9737383222-eulab-document.html


In [19]:
from IPython.display import HTML, display

display(HTML(genomics_html))

Gene,DNA change (c.HGVS),Protein change (p.HGVS),Change type,Zygosity,Classification
BRCA1,NM_007294.3(BRCA1):c.68_69del,p.(Glu23ValfsTer17),deletion,heterozygous,Pathogenic
FBN1,NM_000138.4(FBN1):exon13_to_exon15del,-,copy_number_variation,-,Pathogenic
-,-,-,copy_number_variation,-,Pathogenic
-,-,-,deletion,-,Pathogenic


## Part B — Pathology: no Test Results to merge in

A pathology report has no separate `LAB-5` discrete-results message the way this ctDNA
example does — the `DiagnosticReport` built in section 2 already *is* the whole result,
narrative PDF included. So Part B skips sections 3–6 entirely: no VCF, no variant
`Observation`s, no merge. It reuses the exact same `patient`, `specimen`, `encounter`,
`service_request`, `binary`, `document_reference`, and — importantly — the **original,
unmerged** `diagnostic_report` from section 2, not `merged_diagnostic_report`. Building a
`Composition` around them is the only new work.

### 10. Build the `Composition`

Same `type`/`category`/extension pattern as section 7, but two sections instead of
three — no "Genomic Findings" here, since there are no variant `Observation`s in this
half at all. `laboratory_report_section_html` and `specimen_section_html` are the exact
same strings section 7 built - both messages share the same `report_row`/`specimen`, so
there's nothing pathology-specific to recompute. No `GENOMICS_REPORT_PROFILE` anywhere in
this half either: this `DiagnosticReport` never gained that `meta.profile` or the LOINC
`51969-4` coding, because it's `diagnostic_report`, not `merged_diagnostic_report`.

In [20]:
pathology_composition_fullurl = f"urn:uuid:{uuid4()}"
pathology_composition = {
    "resourceType": "Composition",
    "meta": {"profile": [COMPOSITION_EU_LAB_PROFILE]},
    "extension": [{"url": DIAGNOSTIC_REPORT_REFERENCE_EXT, "valueReference": {"reference": diagnostic_report_fullurl}}],
    "identifier": {"system": "urn:ietf:rfc:3986", "value": f"urn:uuid:{uuid4()}"},
    "status": "final",
    "type": {"coding": [{"system": "http://loinc.org", "code": "11502-2", "display": "Laboratory report"}]},
    "category": [{"coding": [{"system": "http://loinc.org", "code": "26436-6", "display": "Laboratory Studies (set)"}]}],
    "subject": {"reference": patient_fullurl,
                "identifier": {"system": NHS_NUMBER_SYSTEM, "type": {"coding": [{"system": V2_0203, "code": "NH"}]}, "value": report_row["nhs_number"]}},
    "date": report_row["report_datetime"],
    "author": [{"display": GLH_NAME, "identifier": {"system": ODS_SYSTEM, "value": GLH_ODS}, "type": "Organization"}],
    "title": f"{report_row['test_description']} - Laboratory Report",
    "section": [
        {
            "title": "Laboratory Report",
            "code": {"text": report_row["test_description"]},
            "text": {"status": "generated", "div": laboratory_report_section_html},
            "entry": [{"reference": diagnostic_report_fullurl, "type": "DiagnosticReport"}],
        },
        {
            "title": "Specimen",
            "code": {"coding": [{"system": "http://snomed.info/sct", "code": "123038009", "display": "Specimen"}]},
            "text": {"status": "generated", "div": specimen_section_html},
            "entry": [{"reference": specimen_fullurl, "type": "Specimen"}],
        },
    ],
}

validate_resource(pathology_composition, [EU_LAB_IG], [COMPOSITION_EU_LAB_PROFILE])

,severity,code,details,expression
0,information,informational,This element does not match any known slice de...,[Composition.category[0]]
1,information,informational,This element does not match any known slice de...,[Composition.section[0]]
2,information,informational,This element does not match any known slice de...,[Composition.section[1]]
3,warning,code-invalid,None of the codings provided are in the value ...,[Composition.subject.identifier.type]


### 11. Assemble the Document `Bundle`

No `Organization` this time either — nothing in this half references one the way the
variant `Observation`s'/`performer` did in Part A.

In [21]:
pathology_document_bundle = {
    "resourceType": "Bundle",
    "identifier": {"system": "urn:ietf:rfc:3986", "value": f"urn:uuid:{uuid4()}"},
    "timestamp": datetime.now().astimezone().strftime("%Y-%m-%dT%H:%M:%S+00:00"),
    "type": "document",
    "entry": [
        {"fullUrl": pathology_composition_fullurl, "resource": pathology_composition},
        {"fullUrl": patient_fullurl, "resource": patient},
        {"fullUrl": specimen_fullurl, "resource": specimen},
        {"fullUrl": encounter_fullurl, "resource": encounter},
        {"fullUrl": service_request_fullurl, "resource": service_request},
        {"fullUrl": binary_fullurl, "resource": binary},
        {"fullUrl": document_reference_fullurl, "resource": document_reference},
        {"fullUrl": diagnostic_report_fullurl, "resource": diagnostic_report},
    ],
}

pathology_document_filename = "ctdna9737383222-pathology-document.json"
with open("Input/FHIR/R01/" + pathology_document_filename, "w") as f:
    json.dump(pathology_document_bundle, f, indent=2)

print("Saved Input/FHIR/R01/" + pathology_document_filename)

Saved Input/FHIR/R01/ctdna9737383222-pathology-document.json


In [22]:
pathology_outcome_path = Path("Results/FHIR/R01/" + pathology_document_filename + "-OperationOutcome.json")
pathology_composition_index = entry_index(pathology_document_bundle, pathology_composition)
pathology_diagnostic_report_index = entry_index(pathology_document_bundle, diagnostic_report)
pathology_specimen_index = entry_index(pathology_document_bundle, specimen)

validate_bundle(
    "Input/FHIR/R01/" + pathology_document_filename,
    ["package.tgz", EU_LAB_IG],
    [
        (f"Composition:{pathology_composition_index}", COMPOSITION_EU_LAB_PROFILE),
        (f"DiagnosticReport:{pathology_diagnostic_report_index}", DIAGNOSTIC_REPORT_PROFILE),
        (f"Specimen:{pathology_specimen_index}", SPECIMEN_PROFILE),
    ],
    pathology_outcome_path,
)

,severity,code,details,expression
22,error,invalid,Wrong Display Name 'PACKAGE: M4.14 - Non-Small...,[Bundle.entry[7].resource/*DiagnosticReport/nu...
27,information,business-rule,Reference to draft CodeSystem https://fhir.nwg...,[Bundle.entry[7].resource/*DiagnosticReport/nu...
23,information,code-invalid,None of the codings provided are in the value ...,[Bundle.entry[7].resource/*DiagnosticReport/nu...
30,information,not-found,Can't find 'urn:uuid:23b67e47-926e-4382-957d-7...,[Bundle]
13,information,business-rule,Reference to draft CodeSystem https://fhir.nwg...,[Bundle.entry[4].resource/*ServiceRequest/null...
0,information,structure,Details for urn:uuid:2032aa8f-abea-4f1e-8bc3-8...,[Bundle.entry[0].resource/*Composition/null*/....
3,information,informational,This element does not match any known slice de...,[Bundle.entry[0].resource/*Composition/null*/....
5,information,informational,This element does not match any known slice de...,[Bundle.entry[0].resource/*Composition/null*/....
4,information,informational,This element does not match any known slice de...,[Bundle.entry[0].resource/*Composition/null*/....
9,warning,code-invalid,None of the codings provided are in the value ...,[Bundle.entry[3].resource/*Encounter/null*/.su...


### 12. Render the pathology Document as HTML too

Same `document_to_html()` from section 9 — it only reads `Composition`/`Bundle`
structure that's identical between both documents, so nothing pathology-specific was
needed to reuse it.

In [23]:
pathology_html = document_to_html(pathology_document_bundle)
pathology_html_path = "Output/HTML/R01/" + pathology_document_filename.replace(".json", ".html")
with open(pathology_html_path, "w") as f:
    f.write(pathology_html)
print("Saved " + pathology_html_path)

display(HTML(pathology_html))

Saved Output/HTML/R01/ctdna9737383222-pathology-document.html


## Summary

`03`/`04` built the order and report Message `Bundle`s; `05` added a discrete Test
Results message alongside them. This notebook closed the loop: a `Composition` — modelled
on the EU Laboratory IG's own published example rather than guessed from the profile —
turns those Message `Bundle`s into a Document `Bundle`, in the shape NHS England's UGR
Phase 2 and the NHS Pathology FHIR IG expect.

Two documents came out of it, each saved twice — as FHIR under `Input/FHIR/R01/`, and as
a rendered page under `Output/HTML/R01/` (plain, unstyled HTML — that's what
`DocumentToHTML.xslt` itself produces; no CSS was added on top, to keep the rendering
verifiably the official rules and nothing else):

- **`ctdna9737383222-eulab-document.json`** / **`.html`** — the genomics case, merging
  04's Laboratory Report with 05's Test Results into one `DiagnosticReport` and a
  two-section `Composition` (narrative report + a genomic-findings table read straight
  off the variant `Observation`s' own `component[]`).
- **`ctdna9737383222-pathology-document.json`** / **`.html`** — the simpler pathology
  case: the same Laboratory Report resources from section 2, wrapped in a single-section
  `Composition`, no merge.

The two `.html` files are the concrete answer to "what does a FHIR Document actually look
like to a person" — `document_to_html()` (section 9) needed nothing document-specific
to produce either one, because both `Composition`s already carry their own narrative.

The identity problem worth remembering from section 3: two notebooks each minting their
own `urn:uuid`s for what is, in the real world, the same patient, only stays solvable
because `build_observation()` (and any function like it) takes references as *parameters*
rather than hard-coding them. Any conversion pipeline that generates identifiers
independently at each stage needs the same discipline to be mergeable later.